### acceso a openlibrary (no sirve)

In [ ]:
# ======================================================================================
# ACCESO A API DE OPENLIBRARY
# ======================================================================================

import requests
import json
import pandas as pd
from pathlib import Path
import re


# ======================================================================================
# LECTURA DE DATOS
# ======================================================================================

# --- Contribuidores (editor, traductor...)
def leer_brief(isbn):   

    # Llamada para contribuidores
    url = f"http://openlibrary.org/api/volumes/brief/isbn/{isbn}"

    response = requests.get(url)

    if response.status_code == 200:
        api1 = response.json()

        if api1 == []:
            # No encuentra el libro
            return {}
        
        contenido = api1["records"]

        # clave tipo /books/OL9130631M
        clave = list(contenido.keys())[0]
        
        # Edición exacta
        edicion = contenido[clave]["details"]["details"]["edition_name"]

        # Editores
        contribuidores = contenido[clave]["details"]["details"]["contributors"]
        editores = []
        if contribuidores != "":
            for contr in contribuidores:
                if contr['role'] == 'Editor':
                    editores.append(contr["name"])

        # Peso
        peso = contenido[clave]["details"]["details"]['weight']

        # Dimensiones
        dim = contenido[clave]["details"]["details"]['physical_dimensions']

        # Formato
        formato = contenido[clave]["details"]["details"]['physical_format']

        # Categorías
        total_cats = contenido[clave]["data"]["subjects"]
        categorias = []
        for cat_dict in total_cats:
            categorias.append(cat_dict["name"])
        categorias += contenido[clave]["details"]["details"]['subjects']
        # categorias.apply(lambda x: x.translate(str.maketrans({"-":"", "/": "", "&": "and"})).strip())
        

    else: 
        raise Exception(f"Conexión denegada. Status {response.status_code}")
    
    return {'edicion': edicion, 'editores': editores, 'peso': peso, 'dim': dim, 'formato': formato, 'subcategorias': categorias}

# --- Ratings
def leer_ratings(isbn):

    # Llamada para ratings
    url = f"https://openlibrary.org/search.json?isbn={isbn}&fields=rating*"

    response = requests.get(url)

    if response.status_code == 200:
        api2 = response.json()
        ratings = api2["docs"]

    else: 
        raise Exception(f"Conexión denegada. Status {response.status_code}")
    
    return ratings[0] if ratings else {}

# --- Sinopsis (ver como conseguirla de otro sitio)


# ======================================================================================
# CONTROL DE FLUJO INTERACTIVO
# ======================================================================================

def control_flujo_api(ruta_catalogos="data/prueba"): # cambiarlo para que te permita elegir cuántos libros hay que 

    print("Iniciando búsqueda en API...")
    catalogos = [f for f in Path(ruta_catalogos).iterdir() if f.is_file()]

    for ruta_cat in catalogos:
        print(f"\nLeyendo {ruta_cat.name}...")

        with open(ruta_cat, "r", encoding="utf-8") as f:
            catalogo = json.load(f)

        if 'openl' in catalogo[0].keys():
            print("Catálogo completo.")
            continue 
        
        # key = input("Continuar [Y/N]?")    

        # if key=='N':
        #     continue

        print(f"Comenzando con {ruta_cat.name}. Total de libros a buscar: {len(catalogo)+1}...")
        for libro in range(len(catalogo)):
            print(f"[{libro+1}/{len(catalogo)+1}]")
            ean = catalogo[libro]['EAN']

            brief = leer_brief(ean)
            ratings = leer_ratings(ean)

            if isinstance(brief, dict) and isinstance(ratings, dict):
                catalogo[libro].update({
                    **brief,
                    **ratings,
                    'portada': f"https://covers.openlibrary.org/b/isbn/{ean}-L.jpg",
                    'openl': True
                })
            else:
                catalogo[libro]['openl'] = False
                
            print("Catálogo terminado.")

        with open(ruta_cat, "w", encoding="utf-8") as f:
                    json.dump(catalogo, f, ensure_ascii=False, indent=2)


# ======================================================================================
# PUNTO DE ENTRADA
# ======================================================================================

control_flujo_api()


## Modelo

In [ ]:
# prueba scores
import pandas as pd
import numpy as np
import json
from rapidfuzz import fuzz
from src.constants import PESOS_ARQUETIPO

# definición del diccionario de entrada con toda la info

entrada_ej = {
    'busqueda': {
        "titulo_aprox": 'La Celestina', # no tiene porqué ser el título completo/correcto; puede ser None
        "autor": "Fernando de Rojas", # Puede ser None
        "categorias": ['Literatura'], # las del SPI
        "subcategorias": ['Teatro', 'Siglo de Oro'] # las de TTL
    },
    "restricciones":{
        "precio": [0.0, 10000.0],         # De 0 a 10.000 €
        "peso": [0.0, 50000.0],           # De 0 a 50 kg
        "fecha_publicacion": ["1000-01-01", "2099-12-31"],
        "alto_mm": [0.0, 2000.0],         # De 0 a 2 metros
        "ancho_mm": [0.0, 2000.0],        # De 0 a 2 metros
        "grosor_mm": [0.0, 1000.0]        # De 0 a 1 metro
    },
    'perfil': {
        "arquetipo": "estudio_investigacion", # estudio_investigacion, lectura_general, coleccion_regalo, escolar_juvenil
        "flags_adicionales": {
        "es_para_regalo": False,
        "prefiere_ilustrado": False,
        "ed_preferida": None,
        "col_preferida": None,
        "enc_preferida": None
        }
    }
}

def filtro(info_usuario: dict, umbral_similitud: float = 60):
    df = pd.read_parquet("data/gold/gold_df.parquet")
    busqueda = info_usuario['busqueda']
    restricciones = info_usuario['restricciones']

    df_filtrado = df.copy()

    # filtros blandos
    # título y autor
    titulo_query = busqueda["titulo_aprox"]
    autor_query = busqueda["autor"]
    
    if titulo_query:
        scores_titulo = df_filtrado['titulo'].astype(str).apply(
            lambda x: fuzz.partial_ratio(titulo_query.lower(), x.lower())
        )
        df_filtrado = df_filtrado[scores_titulo >= umbral_similitud]

    if autor_query and not df_filtrado.empty:
        scores_autor = df_filtrado['autor'].astype(str).apply(
            lambda x: fuzz.partial_ratio(autor_query.lower(), x.lower())
        )
        df_filtrado = df_filtrado[scores_autor >= umbral_similitud]


    # categorías
    cats_spi = busqueda['categorias']
    subcats_ttl = busqueda['subcategorias']
    
    if cats_spi:
        mask_cats = df_filtrado['categoria_principal'].apply(
        lambda cats_libro: any(c in cats_spi for c in cats_libro) 
        if isinstance(cats_libro, list) else cats_libro in cats_spi)

        df_filtrado = df_filtrado[mask_cats]
        
    if subcats_ttl:
        mask_subcats = df_filtrado['subcategorias'].apply(
            lambda x: any(sub in x for sub in subcats_ttl) if isinstance(x, list) else x in subcats_ttl
        )

        df_filtrado = df_filtrado[mask_subcats]

    # filtros duros
    for col, rest in restricciones.items():
        df_filtrado = df_filtrado[df_filtrado[col].between(rest[0], rest[1])]

    return df_filtrado 


def score_preferencias(df: pd.DataFrame, info_usuario: dict):
    df = df.copy()

    flags = info_usuario['perfil']['flags_adicionales']
    score = pd.Series(0.0, index=df.index)

    if flags["prefiere_ilustrado"] and "es_ilustrado" in df.columns:
        score += df['es_ilustrado'].fillna(False).astype(int) * 0.25

    if flags["ed_preferida"] and "editorial" in df.columns:
        score += (df['editorial'] == flags["ed_preferida"]).astype(int) * 0.30

    if flags["col_preferida"] and "coleccion" in df.columns:
        score += (df['coleccion'] == flags["col_preferida"]).astype(int) * 0.30

    if flags["enc_preferida"] and "encuadernacion" in df.columns:
        score += (df['encuadernacion'] == flags["enc_preferida"]).astype(int) * 0.20

    df['score_perfil'] = score

    return df

def score_arquetipo(df: pd.DataFrame, info_usuario: dict, info_arquetipos: dict = PESOS_ARQUETIPO) -> pd.DataFrame:
    df = df.copy()
    usuario = info_usuario['perfil']['arquetipo']
    arquetipo = info_arquetipos.get(usuario, info_arquetipos['lectura_general'])
    
    score = pd.Series(0.0, index=df.index)

    # 1. Aparato Crítico
    if arquetipo.get("aparato_critico") and "aparato_critico" in df.columns:
        score += df['aparato_critico'].fillna(False).astype(int) * arquetipo["aparato_critico"]

    # 2. Prestigio
    if arquetipo.get("prestigio") and "prestigio_cat" in df.columns:
        max_prestigio = df['prestigio_cat'].max()
        if max_prestigio > 0:
            score += (df['prestigio_cat'] / max_prestigio).fillna(0) * arquetipo["prestigio"]

    # 3. Novedad / Fecha de publicación
    if arquetipo.get("novedad") and "fecha_publicacion" in df.columns:
        años = pd.to_datetime(df['fecha_publicacion'], errors='coerce').dt.year.fillna(2000)
        max_año, min_año = años.max(), años.min()
        if max_año > min_año:
            score_novedad = (años - min_año) / (max_año - min_año)
            score += score_novedad * arquetipo["novedad"]

    # 4. Económico (Precio bajo)
    if arquetipo.get("economico") and "precio" in df.columns:
        max_p, min_p = df['precio'].max(), df['precio'].min()
        if max_p > min_p:
            score_econ = 1 - ((df['precio'] - min_p) / (max_p - min_p))
            score += score_econ * arquetipo["economico"]

    # 5. Formato Premium (Tapa Dura / Ilustrado)
    if arquetipo.get("formato_premium"):
        score_formato = pd.Series(0.0, index=df.index)
        if "encuadernacion" in df.columns:
            es_tapa_dura = df['encuadernacion'].astype(str).str.lower().str.contains("dura|cartoné|piel", na=False)
            score_formato += es_tapa_dura.astype(int) * 0.6
        if "es_ilustrado" in df.columns:
            score_formato += df['es_ilustrado'].fillna(False).astype(int) * 0.4
        
        score += score_formato.clip(upper=1.0) * arquetipo["formato_premium"]

    df['score_arquetipo'] = score
    return df


def scoring_completo(df: pd.DataFrame, info_usuario: dict) -> pd.DataFrame:
    df = df.copy()

    # filtra el df
    df_filtrado = filtro(df)
    if df_filtrado.empty:
        return df_filtrado

    # calcula scores de arquetipo y de preferencias
    df_res = score_arquetipo(df_filtrado, info_usuario)
    df_res = score_preferencias(df_res, info_usuario)
    
    # 3score total
    df['score_perfil'] = (df_res['score_arquetipo'] + df_res['score_preferencias']).round(4)
    
    # 4. Devuelve el DataFrame ordenado de mayor a menor puntuación
    return df.sort_values(by="score_perfil", ascending=False)

In [ ]:
import pandas as pd
import numpy as np
import json
from rapidfuzz import fuzz
from src.constants import PESOS_ARQUETIPO

# definición del diccionario de entrada con toda la info

entrada_ej = {
    'busqueda': {
        "titulo_aprox": 'La Celestina', # no tiene porqué ser el título completo/correcto; puede ser None
        "autor": "Fernando de Rojas", # Puede ser None
        "categorias": ['Literatura'], # las del SPI
        "subcategorias": ['Teatro', 'Siglo de Oro'] # las de TTL
    },
    "restricciones":{
        "precio": [0.0, 10000.0],         # De 0 a 10.000 €
        "peso": [0.0, 50000.0],           # De 0 a 50 kg
        "fecha_publicacion": ["1000-01-01", "2099-12-31"],
        "alto_mm": [0.0, 2000.0],         # De 0 a 2 metros
        "ancho_mm": [0.0, 2000.0],        # De 0 a 2 metros
        "grosor_mm": [0.0, 1000.0]        # De 0 a 1 metro
    },
    'perfil': {
        "arquetipo": "estudio_investigacion", # estudio_investigacion, lectura_general, coleccion_regalo, escolar_juvenil
        "flags_adicionales": {
        "es_para_regalo": False,
        "prefiere_ilustrado": False,
        "ed_preferida": None,
        "col_preferida": None,
        "enc_preferida": None
        }
    }
}

In [ ]:
# ===================================================================================
# MODELO
# ===================================================================================

# PASO 1: FILTRO
def filtro(info_usuario: dict, umbral_similitud: float = 60):
    df = pd.read_parquet("data/gold/gold_df.parquet")
    busqueda = info_usuario['busqueda']
    restricciones = info_usuario['restricciones']

    df_filtrado = df.copy()

    # filtros blandos
    # título y autor
    titulo_query = busqueda["titulo_aprox"]
    autor_query = busqueda["autor"]
    
    if titulo_query:
        scores_titulo = df_filtrado['titulo'].astype(str).apply(
            lambda x: fuzz.partial_ratio(titulo_query.lower(), x.lower())
        )
        df_filtrado = df_filtrado[scores_titulo >= umbral_similitud]

    if autor_query and not df_filtrado.empty:
        scores_autor = df_filtrado['autor'].astype(str).apply(
            lambda x: fuzz.partial_ratio(autor_query.lower(), x.lower())
        )
        df_filtrado = df_filtrado[scores_autor >= umbral_similitud]


    # categorías
    cats_spi = busqueda['categorias']
    subcats_ttl = busqueda['subcategorias']
    
    if cats_spi:
        mask_cats = df_filtrado['categoria_principal'].apply(
        lambda cats_libro: any(c in cats_spi for c in cats_libro) 
        if isinstance(cats_libro, list) else cats_libro in cats_spi)

        df_filtrado = df_filtrado[mask_cats]
        
    if subcats_ttl:
        mask_subcats = df_filtrado['subcategorias'].apply(
            lambda x: any(sub in x for sub in subcats_ttl) if isinstance(x, list) else x in subcats_ttl
        )

        df_filtrado = df_filtrado[mask_subcats]

    # filtros duros
    for col, rest in restricciones.items():
        df_filtrado = df_filtrado[df_filtrado[col].between(rest[0], rest[1])]

    return df_filtrado

# PASO 2: CÁLCULO DE PESOS
CRITERIOS = [
    "indice_portabilidad",
    "indice_compacidad",
    "indice_prestancia",
    "aparato_critico",
    "prestigio_cat"
]
def calcular_vector_ahp(matriz: np.ndarray, columnas: list = CRITERIOS):
    """
    Calcula el vector de pesos normalizado y el Ratio de Consistencia (CR).
    """
    n = matriz.shape[0]
    
    # 1. Autovalores y autovectores
    autovalores, autovectores = np.linalg.eig(matriz)
    max_idx = np.argmax(np.real(autovalores))
    lambda_max = np.real(autovalores[max_idx])
    
    # 2. Vector propio principal normalizado (Pesos w_j)
    weights = np.real(autovectores[:, max_idx])
    weights = weights / np.sum(weights)
    
    # 3. Ratio de Consistencia (CR de Saaty)
    ci = (lambda_max - n) / (n - 1)
    ri_5 = 1.12  # Valor aleatorio de Saaty para n = 5
    cr = ci / ri_5
    
    dict_pesos = dict(zip(columnas, np.round(weights, 4)))
    
    return dict_pesos, cr

# PASO 3: MODELO TOPSIS
def ejecutar_topsis(df: pd.DataFrame, pesos_dict: dict) -> pd.DataFrame:
    df_res = df.copy()
    cols = list(pesos_dict.keys())
    
    # 1. Matriz de decisión
    X = df_res[cols].astype(float).values
    
    # 2. Normalización Vectorial
    normas = np.sqrt((X**2).sum(axis=0))
    normas[normas == 0.0] = 1.0
    X_norm = X / normas
    
    # 3. Ponderación con los pesos AHP
    weights = np.array([pesos_dict[c] for c in cols])
    X_weighted = X_norm * weights
    
    # 4. Solución Ideal Positiva (A+) e Ideal Negativa (A-)
    # 'precio' es costo (minimizar); los demás son beneficios (maximizar)
    ideal_pos = []
    ideal_neg = []
    
    for i, col in enumerate(cols):
        if col == "precio":
            ideal_pos.append(X_weighted[:, i].min())
            ideal_neg.append(X_weighted[:, i].max())
        else:
            ideal_pos.append(X_weighted[:, i].max())
            ideal_neg.append(X_weighted[:, i].min())
            
    ideal_pos = np.array(ideal_pos)
    ideal_neg = np.array(ideal_neg)
    
    # 5. Distancias Euclídeas
    d_pos = np.sqrt(((X_weighted - ideal_pos)**2).sum(axis=1))
    d_neg = np.sqrt(((X_weighted - ideal_neg)**2).sum(axis=1))
    
    # 6. Cercanía Relativa (Score TOPSIS de 0 a 1)
    df_res["score_topsis"] = d_neg / (d_pos + d_neg)
    
    return df_res.sort_values(by="score_topsis", ascending=False)

# Ejecutar recomendación
df_ranking = ejecutar_topsis(df_libros, pesos_estudio)

print("\n" + "=" * 60)
print("3. RESULTADO FINAL DEL RANKING TOPSIS")
print("=" * 60)
print(df_ranking[["editorial", "score_topsis", "aparato_critico", "prestigio_cat", "precio"]])